In [ ]:
%pip install -q pandas requests tzdata

In [ ]:
from pathlib import Path
from datetime import time as clock_time
import hashlib
import json
import math
import time

import pandas as pd
import requests

# ============================================================
# SETTINGS
# ============================================================

PART_NUMBER = 5           # Choose 1–7.
TOWNS_PER_PART = 3

START_DATE = "2023-06-01"
END_DATE = "2026-08-02"

TIMEZONE = "Asia/Colombo"
START_TIME = clock_time(6, 0)
END_TIME = clock_time(19, 0)

REQUEST_PAUSE_SECONDS = 10

# # PC: use a relative folder or your preferred absolute path.
# OUTPUT_FOLDER = Path("weather_downloads")

OUTPUT_FOLDER = Path("/content/weather_downloads")

# COLAB: to preserve downloads when the session disconnects,
# uncomment these three lines and use Google Drive.
#
# from google.colab import drive
# drive.mount("/content/drive")
# OUTPUT_FOLDER = Path("/content/drive/MyDrive/solar_weather")

API_URL = "https://archive-api.open-meteo.com/v1/archive"

# ============================================================
# UNIQUE LOCATIONS FROM YOUR LIST
# ============================================================

locations = pd.DataFrame(
    [
        ("Habaraduwa",        "Galle",       5.9977, 80.2958),
        ("Wadduwa",          "Kalutara",    6.6640, 79.9305),
        ("Ambalangoda",      "Galle",       6.2353, 80.0550),
        ("Galle",            "Galle",       6.0329, 80.2168),
        ("Ahungalla",        "Galle",       6.3117, 80.0345),
        ("Ahangama",         "Galle",       5.9726, 80.3653),
        ("Hikkaduwa",        "Galle",       6.1396, 80.1030),
        ("Pitiwella",        "Galle",       6.0712, 80.1691),
        ("Ambalantota",      "Hambantota",  6.1225, 81.0239),
        ("Payagala",         "Kalutara",    6.5683, 79.9825),
        ("Matugama",         "Kalutara",    6.5222, 80.1147),
        ("Sella Kataragama", "Monaragala",  6.4442, 81.3090),
        ("Beliatta",         "Hambantota",  6.0394, 80.7445),
        ("Uhana",            "Ampara",      7.3323, 81.6781),
        ("Akmeemana",        "Galle",       6.0664, 80.2589),
        ("Balapitiya",       "Galle",       6.2731, 80.0392),
        ("Weligama",         "Matara",      5.9722, 80.4287),
        ("Kalametiya",       "Hambantota",  6.0839, 80.9404),
        ("Tangalle",         "Hambantota",  6.0243, 80.7942),
        ("Kalutara",         "Kalutara",    6.5854, 79.9607),
        ("Weniyarawela",     "Galle",       6.1865, 80.1652),
    ],
    columns=["town", "district", "latitude", "longitude"],
).sort_values(
    ["town", "latitude", "longitude"]
).reset_index(drop=True)

locations["location_id"] = [
    f"LK{i + 1:03d}" for i in range(len(locations))
]

locations["part_number"] = (
    locations.index // TOWNS_PER_PART + 1
)

# ============================================================
# 45 HOURLY WEATHER VARIABLES
# ============================================================

WEATHER_VARIABLES = """
temperature_2m
relative_humidity_2m
dew_point_2m
apparent_temperature
precipitation
rain
snowfall
snow_depth
weather_code
pressure_msl
surface_pressure
cloud_cover
cloud_cover_low
cloud_cover_mid
cloud_cover_high
et0_fao_evapotranspiration
vapour_pressure_deficit
wind_speed_10m
wind_speed_100m
wind_direction_10m
wind_direction_100m
wind_gusts_10m
soil_temperature_0_to_7cm
soil_temperature_7_to_28cm
soil_temperature_28_to_100cm
soil_temperature_100_to_255cm
soil_moisture_0_to_7cm
soil_moisture_7_to_28cm
soil_moisture_28_to_100cm
soil_moisture_100_to_255cm
boundary_layer_height
wet_bulb_temperature_2m
total_column_integrated_water_vapour
is_day
sunshine_duration
shortwave_radiation
direct_radiation
diffuse_radiation
direct_normal_irradiance
terrestrial_radiation
shortwave_radiation_instant
direct_radiation_instant
diffuse_radiation_instant
direct_normal_irradiance_instant
terrestrial_radiation_instant
""".split()

print("Download parts:")
display(
    locations.groupby("part_number")["town"]
    .agg(", ".join)
    .rename("towns")
    .to_frame()
)

In [ ]:
# ============================================================
# VALIDATE SETTINGS
# ============================================================

start_date = pd.Timestamp(START_DATE).normalize()
end_date = pd.Timestamp(END_DATE).normalize()

number_of_parts = math.ceil(len(locations) / TOWNS_PER_PART)

if start_date > end_date:
    raise ValueError("START_DATE must not exceed END_DATE")

if not 1 <= PART_NUMBER <= number_of_parts:
    raise ValueError(
        f"PART_NUMBER must be between 1 and {number_of_parts}"
    )

if START_TIME > END_TIME:
    raise ValueError("START_TIME must not exceed END_TIME")

if REQUEST_PAUSE_SECONDS < 10:
    raise ValueError("Keep at least 10 seconds between requests")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

CACHE_FOLDER = OUTPUT_FOLDER / "cache"
CACHE_FOLDER.mkdir(exist_ok=True)

locations.to_csv(
    OUTPUT_FOLDER / "locations_and_parts.csv",
    index=False,
    encoding="utf-8-sig",
)

part_locations = locations.loc[
    locations["part_number"].eq(PART_NUMBER)
].copy()

days_requested = (end_date - start_date).days + 1
requests_per_town = math.ceil(days_requested / 14)

estimated_requests = requests_per_town * len(part_locations)

estimated_call_units = (
    estimated_requests
    * max(1, len(WEATHER_VARIABLES) / 10)
)

print(f"Part: {PART_NUMBER}/{number_of_parts}")
print("Towns:", ", ".join(part_locations["town"]))
print(f"Dates: {START_DATE} through {END_DATE}")
print(f"Maximum HTTP requests without cache: {estimated_requests}")
print(f"Estimated API call units: up to {estimated_call_units:,.0f}")
print(f"Delay: {REQUEST_PAUSE_SECONDS} seconds per request")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def save_json(path, content):
    temporary = path.with_suffix(path.suffix + ".tmp")

    temporary.write_text(
        json.dumps(content, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    temporary.replace(path)


def save_csv(frame, path):
    temporary = path.with_suffix(".csv.tmp")

    frame.to_csv(
        temporary,
        index=False,
        encoding="utf-8-sig",
    )

    temporary.replace(path)


def date_chunks(start, end):
    """Produce non-overlapping periods of at most 14 days."""
    current = start

    while current <= end:
        stop = min(
            current + pd.Timedelta(days=13),
            end,
        )

        yield current, stop
        current = stop + pd.Timedelta(days=1)


def validate_response(payload, start, end):
    if payload.get("error"):
        raise ValueError(
            payload.get("reason", "Open-Meteo API error")
        )

    hourly = payload.get("hourly", {})

    expected_times = pd.date_range(
        start=start,
        end=end + pd.Timedelta(days=1),
        freq="h",
        inclusive="left",
    )

    actual_times = pd.DatetimeIndex(
        pd.to_datetime(hourly.get("time", []))
    )

    if not actual_times.equals(expected_times):
        raise ValueError(
            "Incomplete or unexpected hourly weather timeline"
        )

    if payload.get("utc_offset_seconds") != 19800:
        raise ValueError("Unexpected timezone offset")

    for variable in WEATHER_VARIABLES:
        if variable not in hourly:
            raise ValueError(
                f"Missing weather variable: {variable}"
            )

        if len(hourly[variable]) != len(expected_times):
            raise ValueError(
                f"Incomplete weather variable: {variable}"
            )

    return payload


def download_chunk(session, params, start, end):
    cache_key = hashlib.sha256(
        json.dumps(params, sort_keys=True).encode()
    ).hexdigest()

    cache_file = CACHE_FOLDER / f"{cache_key}.json"

    if cache_file.exists():
        payload = json.loads(
            cache_file.read_text(encoding="utf-8")
        )
        return validate_response(payload, start, end)

    max_attempts = 6

    for attempt in range(max_attempts):
        time.sleep(REQUEST_PAUSE_SECONDS)

        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=(15, 180),
            )
        except (
            requests.Timeout,
            requests.ConnectionError,
            requests.exceptions.ChunkedEncodingError,
        ) as error:
            if attempt == max_attempts - 1:
                raise

            delay = min(120, 20 * (attempt + 1))
            print(
                f"{type(error).__name__}; retrying in {delay}s",
                flush=True,
            )
            time.sleep(delay)
            continue

        # Retry temporary server errors and rate limits.
        if response.status_code == 429 or response.status_code >= 500:
            if attempt == max_attempts - 1:
                response.raise_for_status()

            retry_after = response.headers.get("Retry-After", "")
            delay = (
                max(60, float(retry_after))
                if retry_after.isdigit()
                else 60
            )

            print(
                f"HTTP {response.status_code}; retrying in {delay:.0f}s",
                flush=True,
            )
            time.sleep(delay)
            continue

        # Stop on other HTTP errors, such as invalid parameters.
        if not response.ok:
            raise RuntimeError(
                f"HTTP {response.status_code}: "
                f"{response.text[:500]}"
            )

        # Retry empty, truncated, or non-JSON successful responses.
        try:
            payload = response.json()
        except ValueError as error:
            content_type = response.headers.get(
                "Content-Type", "unknown"
            )

            if attempt == max_attempts - 1:
                raise RuntimeError(
                    "Repeated invalid JSON response. "
                    f"HTTP {response.status_code}; "
                    f"Content-Type: {content_type}; "
                    f"Response preview: {response.text[:200]!r}"
                ) from error

            delay = min(120, 20 * (attempt + 1))

            print(
                f"Invalid JSON response "
                f"(HTTP {response.status_code}, {content_type}); "
                f"retrying in {delay}s",
                flush=True,
            )
            time.sleep(delay)
            continue

        # Only cache complete, validated responses.
        payload = validate_response(payload, start, end)
        save_json(cache_file, payload)

        return payload

    raise RuntimeError("Download attempts exhausted")


# ============================================================
# DOWNLOAD
# ============================================================

frames = []
request_metadata = []

with requests.Session() as session:
    for location in part_locations.to_dict("records"):
        print(
            f"\nDownloading: {location['town']}",
            flush=True,
        )

        for chunk_start, chunk_end in date_chunks(
            start_date, end_date
        ):
            params = {
                "latitude": location["latitude"],
                "longitude": location["longitude"],
                "start_date": chunk_start.strftime("%Y-%m-%d"),
                "end_date": chunk_end.strftime("%Y-%m-%d"),
                "hourly": ",".join(WEATHER_VARIABLES),
                "timezone": TIMEZONE,
                "temperature_unit": "celsius",
                "wind_speed_unit": "ms",
                "precipitation_unit": "mm",
            }

            payload = download_chunk(
                session,
                params,
                chunk_start,
                chunk_end,
            )

            frame = pd.DataFrame(payload["hourly"])

            local_time = (
                pd.to_datetime(frame.pop("time"))
                .dt.tz_localize(TIMEZONE)
            )

            frame.insert(
                0, "timestamp_local", local_time
            )

            frame.insert(
                1,
                "timestamp_utc",
                local_time.dt.tz_convert("UTC"),
            )

            # Requested location: use these coordinates when
            # matching to your solar dataset.
            for column in [
                "location_id",
                "town",
                "district",
                "latitude",
                "longitude",
            ]:
                frame[column] = location[column]

            frame["part_number"] = PART_NUMBER

            # Actual weather grid coordinates can differ.
            frame["weather_grid_latitude"] = payload["latitude"]
            frame["weather_grid_longitude"] = payload["longitude"]
            frame["weather_elevation_m"] = payload.get("elevation")

            frame["weather_source"] = (
                "Open-Meteo Historical Weather"
            )
            frame["weather_model_selection"] = "best_match"

            frames.append(frame)

            request_metadata.append({
                "location_id": location["location_id"],
                "request": params,
                "units": payload.get("hourly_units", {}),
                "weather_grid_latitude": payload["latitude"],
                "weather_grid_longitude": payload["longitude"],
            })

            print(
                f"  Ready: {chunk_start.date()} "
                f"through {chunk_end.date()}",
                flush=True,
            )


# ============================================================
# VALIDATE AND SAVE
# ============================================================

full = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["location_id", "timestamp_local"])
    .reset_index(drop=True)
)

if full.duplicated(
    ["location_id", "timestamp_utc"]
).any():
    raise ValueError(
        "Unexpected duplicate location/timestamp pairs"
    )

local_clock = full["timestamp_local"].dt.time

daytime = full.loc[
    (local_clock >= START_TIME)
    & (local_clock <= END_TIME)
].copy()

missing_summary = (
    full.groupby("location_id")[WEATHER_VARIABLES]
    .agg(lambda column: column.isna().sum())
    .reset_index()
)

prefix = (
    f"weather_part_{PART_NUMBER:02d}_"
    f"{START_DATE}_{END_DATE}"
)

window_label = (
    START_TIME.strftime("%H%M")
    + "_"
    + END_TIME.strftime("%H%M")
)

# Explicit ISO timestamps, e.g.:
# 2023-06-01 16:00:00+05:30
for frame in (full, daytime):
    for column in [
        "timestamp_local",
        "timestamp_utc",
    ]:
        frame[column] = frame[column].map(
            lambda value: value.isoformat(sep=" ")
        )

daytime_file = (
    OUTPUT_FOLDER / f"{prefix}_{window_label}.csv"
)
full_day_file = (
    OUTPUT_FOLDER / f"{prefix}_full_day.csv"
)

save_csv(daytime, daytime_file)

# Keep surrounding hours for later interpolation or
# alignment of preceding-hour weather variables.
save_csv(full, full_day_file)

save_csv(
    missing_summary,
    OUTPUT_FOLDER / f"{prefix}_missing_values.csv",
)

save_json(
    OUTPUT_FOLDER / f"{prefix}_metadata.json",
    {
        "part_number": PART_NUMBER,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "timezone": TIMEZONE,
        "time_window": window_label,
        "temporal_resolution": "hourly",
        "variables": WEATHER_VARIABLES,
        "locations": part_locations.to_dict("records"),
        "requests": request_metadata,
        "notes": (
            "Historical weather estimates, not archived "
            "day-ahead forecasts. No five-minute interpolation "
            "has been applied. Missing values are retained."
        ),
        "attribution": (
            "Weather data by Open-Meteo.com, CC BY 4.0. "
            "Underlying sources: see Open-Meteo documentation."
        ),
    },
)

print(f"\nFinished part {PART_NUMBER}")
print(f"Selected-hours rows: {len(daytime):,}")
print(f"Main CSV: {daytime_file.resolve()}")
print(f"Full-day CSV: {full_day_file.resolve()}")

In [ ]:
from google.colab import files

files.download(str(daytime_file))

In [ ]:
files.download(
    str(OUTPUT_FOLDER / f"{prefix}_metadata.json")
)